### start stop spark

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

spark = SparkSession.builder \
    .appName("SparkExample") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

### import tables

In [ ]:
# Параметры подключения к PostgreSQL public.shops 

jdbc_url = "jdbc:postgresql://postgres_source:5432/source" 
table_name = "public.shops" 
db_user = os.getenv("POSTGRES_USER") 
db_password = os.getenv("POSTGRES_PASSWORD")


shops_df = spark.read.format("jdbc")\
		.option("url", jdbc_url)\
		.option("user", db_user)\
		.option("password", db_password)\
		.option("dbtable", table_name)\
		.option("fetchsize", 1000)\
		.option("driver", "org.postgresql.Driver")\
		.load() 


shops_df.show()

In [ ]:
# ⬇️ Параметры подключения к PostgreSQLpublic.shop_timezone 


jdbc_url = "jdbc:postgresql://postgres_source:5432/source" 
table_name = "public.shop_timezone" 
db_user = os.getenv("POSTGRES_USER") 
db_password = os.getenv("POSTGRES_PASSWORD") 


shop_timezone_df = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("user", db_user) \
    .option("password", db_password) \
    .option("dbtable", table_name) \
    .option("fetchsize", 1000) \
    .option("driver", "org.postgresql.Driver") \
    .load()



shop_timezone_df.show()

### ....

In [7]:
# Регистрируем DataFrame-ы как временные вьюхи
shops_df.createOrReplaceTempView("shops")
shop_timezone_df.createOrReplaceTempView("shop_timezone")

In [ ]:
shops_df.show(3)

In [ ]:
shop_timezone_df.show(3)

In [ ]:
names = shops_df.select(
    F.col("st_id").cast(IntegerType()).alias("st_id"),
    "shop_name"
).where("st_id IS NOT NULL")

timezones = shop_timezone_df.select(
    F.col("plant").cast(IntegerType()).alias("st_id"),
    F.when(
        (F.col("time_zone") == "") | (F.col("time_zone").isNull()),
        3
    ).otherwise(
        F.regexp_replace(F.col("time_zone"), "RUS0", "").cast(IntegerType())
    ).alias("tz_code")
).where("st_id IS NOT NULL")

result_df = names.join(
    F.broadcast(timezones),
    on="st_id",
    how="inner"
).select("st_id", "shop_name", "tz_code") \
 .orderBy("st_id")

result_df.show()



In [ ]:
spark.stop()